In [1]:
!pip install -q fastapi uvicorn scikit-learn pandas joblib pyngrok nest_asyncio requests

In [2]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

joblib.dump(model, "model.pkl")
print("model.pkl saved successfully")

Accuracy: 1.0
model.pkl saved successfully


In [3]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

model = joblib.load("model.pkl")

app = FastAPI(title="Iris Prediction API")

class IrisFeatures(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.get("/")
def home():
    return {"message": "Iris Prediction API is running"}

@app.post("/predict")
def predict(features: IrisFeatures):
    data = np.array([[
        features.sepal_length,
        features.sepal_width,
        features.petal_length,
        features.petal_width
    ]])

    prediction = model.predict(data)[0]
    species = ["setosa", "versicolor", "virginica"][prediction]

    return {
        "prediction": int(prediction),
        "species": species
    }

In [4]:
from pyngrok import ngrok

ngrok.set_auth_token("3Ii5Wirnlh3Dq5SCvTLPgN0hJ8H_4hUWJWq277hYjcSkoZKRX")

print("ngrok authenticated successfully")

ngrok authenticated successfully


In [5]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()

print("FastAPI server started on port 8000")

FastAPI server started on port 8000


In [6]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://aflame-unnamed-careless.ngrok-free.dev" -> "http://localhost:8000"


In [7]:
import requests

url = public_url.public_url + "/predict"

data = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}

response = requests.post(url, json=data)

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:     35.225.210.231:0 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Response: {'prediction': 0, 'species': 'setosa'}


In [10]:
data = {
    "sepal_length": 6.7,
    "sepal_width": 3.1,
    "petal_length": 4.7,
    "petal_width": 1.5
}

response = requests.post(
    public_url.public_url + "/predict",
    json=data
)

print("Status Code:", response.status_code)
print("Response:", response.json())


INFO:     35.225.210.231:0 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Response: {'prediction': 1, 'species': 'versicolor'}


In [9]:
data = {
    "sepal_length": 6.3,
    "sepal_width": 3.3,
    "petal_length": 6.0,
    "petal_width": 2.5
}

response = requests.post(
    public_url.public_url + "/predict",
    json=data
)

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:     35.225.210.231:0 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Response: {'prediction': 2, 'species': 'virginica'}


In [11]:
response = requests.get(public_url.public_url + "/")

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:     35.225.210.231:0 - "GET / HTTP/1.1" 200 OK
Status Code: 200
Response: {'message': 'Iris Prediction API is running'}


In [12]:
app_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

model = joblib.load("model.pkl")

app = FastAPI()

class IrisFeatures(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.get("/")
def home():
    return {"message": "API Running"}

@app.post("/predict")
def predict(features: IrisFeatures):
    data = np.array([[
        features.sepal_length,
        features.sepal_width,
        features.petal_length,
        features.petal_width
    ]])

    pred = model.predict(data)[0]
    species = ["setosa", "versicolor", "virginica"][pred]

    return {
        "prediction": int(pred),
        "species": species
    }
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created successfully")

app.py created successfully


In [13]:
import os

print("app.py exists:", os.path.exists("app.py"))

app.py exists: True


In [14]:
dockerfile = '''
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
COPY model.pkl .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

requirements = '''
fastapi
uvicorn
scikit-learn
numpy
joblib
pydantic
'''

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("Dockerfile and requirements.txt created")

Dockerfile and requirements.txt created


In [15]:
import os

print("Dockerfile:", os.path.exists("Dockerfile"))
print("requirements.txt:", os.path.exists("requirements.txt"))

Dockerfile: True
requirements.txt: True


In [16]:
k8s_yaml = '''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-api
spec:
  replicas: 1
  selector:
    matchLabels:
      app: iris-api
  template:
    metadata:
      labels:
        app: iris-api
    spec:
      containers:
      - name: iris-api
        image: iris-api:latest
        ports:
        - containerPort: 8000
---
apiVersion: v1
kind: Service
metadata:
  name: iris-api-service
spec:
  selector:
    app: iris-api
  ports:
  - protocol: TCP
    port: 80
    targetPort: 8000
  type: LoadBalancer
'''

with open("deployment.yaml", "w") as f:
    f.write(k8s_yaml)

print("deployment.yaml created successfully")

deployment.yaml created successfully


In [17]:
print("deployment.yaml exists:", os.path.exists("deployment.yaml"))

deployment.yaml exists: True


In [18]:
from google.colab import files

files.download("app.py")
files.download("model.pkl")
files.download("Dockerfile")
files.download("requirements.txt")
files.download("deployment.yaml")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>